# Day 59：毕业设计开发 Day 1：项目骨架、配置中心与数据库底座（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 9 周**：星辰智脑毕业设计全栈开发与上线  
> **GitLab 工单**：`STARMIND-059` ｜ **开发分支**：`feat/day59-capstone-infra-base`  
> **配套 HTML5 课件**：`courseware/day59/index.html` ｜ **独立版本快照**：`versions/day59/`

---

## 🎙️ 老陈开场旁白

早啊各位！毕业设计开发 Day 1，咱们先把地基打成钢筋混凝土。很多同学写项目喜欢在代码里到处 `os.getenv`、到处 `print()`、数据库连表结构都随便拿 JSON 糊弄，结果到 Day 62 联调时一报错根本不知道哪条请求炸的。今天老陈带你按大厂标准把四大底座一次建齐：1. Pydantic Settings 强类型配置中心；2. 自带 `trace_id` 的 JSON 结构化日志与统一业务错误码；3. SQLAlchemy 2.0 持久化模型（会话表、消息表、知识文档表、审计日志表）与仓储层（Repository）。地基稳了，后面五天就像搭乐高一样丝滑！

### 承上启下
- **昨日基线**：昨天我们冻结了《星辰智脑 v1.0》的 PRD 与六大模块架构评审，今天正式敲响毕业设计第一行生产代码。
- **明日预告**：明天（Day 60）我们在今天的数据库和日志底座之上，开发星辰智脑核心 RAG 知识库主链路：混合检索（BM25+向量）、Rerank 重排与强制引用溯源。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day59'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-059）

星辰智脑 v1.0 需在开发首日完成工程底座建设，为后续 RAG、多 Agent、FastAPI 接口与运维监控提供统一的配置读取、全链路追踪日志、错误码规范及关系型数据持久化能力。

**今日迭代目标：**
- 基于 `pydantic-settings` 实现启动期强校验配置中心（支持 `.env` 与环境变量覆盖）。
- 实现自定义业务异常体系（`StarMindError` + 规范错误码）与 JSON 格式结构化日志器（支持 `trace_id` 注入）。
- 基于 `SQLAlchemy 2.0` 建立 `sessions`、`messages`、`audit_logs` 三张核心表及 Repository CRUD 封装。

**验收标准：**
- 配置中心对非法端口或超界阈值在初始化时立刻抛出 `ValidationError`。
- 结构化日志器输出合法 JSON 字符串，包含 `timestamp`、`level`、`trace_id`、`event` 字段。
- SQLAlchemy 底座在内存/本地 SQLite 上完整跑通会话创建、多轮消息写入与审计记录查询。

---

## 📚 课堂核心技术笔记

### 1. 1. Pydantic Settings 的 Fail-Fast 价值

使用 `pydantic_settings.BaseSettings` 声明配置项，配合 `Field(ge=..., le=...)` 约束范围，服务启动时只要有非法环境变量立刻报错终止，绝不让带病配置流入线上。

> **⚠️ 老陈避坑**：坑点：在测试里直接硬编码修改全局 settings 单例导致用例之间互相污染，推荐提供工厂函数或依赖注入。

### 2. 2. 为什么大厂一律要求 JSON 结构化日志？

非结构化的 `print(f'用户 {uid} 报错 {e}')` 无法被日志平台做精确聚合统计；JSON 日志可以直接按 `level=='ERROR'`、`latency_ms>1000` 或 `trace_id=='tr-123'` 秒级检索和配置监控大盘。

> **⚠️ 老陈避坑**：坑点：把用户的明文手机号或 API Key 直接打进了 JSON 日志，引发次生数据泄露。日志打印前务必脱敏。

### 3. 3. SQLAlchemy 2.0 现代写法规范

使用 `DeclarativeBase` + `Mapped[T]` + `mapped_column()` 的强类型声明方式，搭配 `with Session(engine) as db:` 上下文管理事务，告别旧版 1.x 的隐式连接泄露问题。

> **⚠️ 老陈避坑**：坑点：在多线程/FastAPI 异步接口中复用同一个 Session 实例导致并发事务冲突，每次请求必须使用独立 Session。


## 💻 实操步骤 1：步骤一：实现强类型配置中心、统一异常体系与 JSON 结构化日志器

- **目标文件**：`versions/day59/src/day59_core_infra.py`
- **老陈代码旁白**：先搭好 `core` 层三剑客：配置中心、错误码异常基类和带 `trace_id` 的 JSON 日志输出器。


In [ ]:
"""
Day 59 - 模块一：星辰智脑核心底座（配置中心 + 统一异常 + JSON结构化日志）
"""
from __future__ import annotations

import json
import uuid
from datetime import datetime, timezone
from typing import Any, Dict, Optional

from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict


class StarMindSettings(BaseSettings):
    """星辰智脑强类型配置中心"""
    model_config = SettingsConfigDict(env_prefix="STARMIND_", extra="ignore")

    app_name: str = "StarMind-Enterprise"
    env: str = "dev"
    database_url: str = "sqlite:///:memory:"
    default_model: str = "starmind-7b-local"
    daily_budget_cny: float = Field(default=100.0, gt=0.0)
    rag_top_k: int = Field(default=5, ge=1, le=20)


class StarMindError(Exception):
    """统一业务异常基类"""

    def __init__(self, code: int, message: str, http_status: int = 400, trace_id: str = "") -> None:
        super().__init__(message)
        self.code = code
        self.message = message
        self.http_status = http_status
        self.trace_id = trace_id or new_trace_id()

    def to_dict(self) -> Dict[str, Any]:
        return {
            "code": self.code,
            "message": self.message,
            "trace_id": self.trace_id,
        }


def new_trace_id() -> str:
    return f"tr-{uuid.uuid4().hex[:12]}"


def format_json_log(
    level: str,
    event: str,
    trace_id: str,
    extra: Optional[Dict[str, Any]] = None,
) -> str:
    """生成单行 JSON 结构化日志，适配 ELK / 阿里云 SLS"""
    payload: Dict[str, Any] = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "level": level.upper(),
        "trace_id": trace_id,
        "event": event,
    }
    if extra:
        payload["extra"] = extra
    return json.dumps(payload, ensure_ascii=False)


if __name__ == "__main__":
    cfg = StarMindSettings()
    tid = new_trace_id()
    print(format_json_log("info", "app_boot", tid, {"app": cfg.app_name, "top_k": cfg.rag_top_k}))
    err = StarMindError(4291, "部门日配额已耗尽触发熔断", http_status=429, trace_id=tid)
    print("异常封包:", err.to_dict())


### 🔍 步骤 1 代码深度解析

`StarMindSettings` 支持通过 `STARMIND_` 前缀环境变量覆盖；`StarMindError` 将业务错误码、HTTP 状态码与 `trace_id` 打包统一，前端拿到就能精准展示与反馈工单。

**预期运行输出参考：**
```text
{"timestamp": "...", "level": "INFO", "trace_id": "tr-...", "event": "app_boot", "extra": {"app": "StarMind-Enterprise", "top_k": 5}}
异常封包: {'code': 4291, 'message': '部门日配额已耗尽触发熔断', 'trace_id': 'tr-...'}
```


## 💻 实操步骤 2：步骤二：实现 SQLAlchemy 2.0 持久化模型与 Repository 仓储层

- **目标文件**：`versions/day59/src/day59_db_repository.py`
- **老陈代码旁白**：接着用 SQLAlchemy 2.0 建好 `chat_sessions`、`chat_messages`、`audit_logs` 三张表和仓储类，支撑后续多轮对话记忆和合规审计。


In [ ]:
"""
Day 59 - 模块二：SQLAlchemy 2.0 持久化模型与 Repository 仓储底座
"""
from __future__ import annotations

from datetime import datetime, timezone
from typing import Dict, List

from sqlalchemy import DateTime, ForeignKey, Integer, String, Text, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column


class Base(DeclarativeBase):
    pass


class ChatSession(Base):
    __tablename__ = "chat_sessions"

    id: Mapped[str] = mapped_column(String(64), primary_key=True)
    tenant_id: Mapped[str] = mapped_column(String(64), index=True)
    title: Mapped[str] = mapped_column(String(200), default="新会话")
    created_at: Mapped[datetime] = mapped_column(
        DateTime(timezone=True), default=lambda: datetime.now(timezone.utc)
    )


class ChatMessage(Base):
    __tablename__ = "chat_messages"

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    session_id: Mapped[str] = mapped_column(ForeignKey("chat_sessions.id"), index=True)
    role: Mapped[str] = mapped_column(String(32))
    content: Mapped[str] = mapped_column(Text)
    tokens: Mapped[int] = mapped_column(Integer, default=0)


class AuditLog(Base):
    __tablename__ = "audit_logs"

    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    trace_id: Mapped[str] = mapped_column(String(64), index=True)
    tenant_id: Mapped[str] = mapped_column(String(64))
    action: Mapped[str] = mapped_column(String(64))
    status: Mapped[str] = mapped_column(String(32))


class StarMindRepository:
    """统一数据访问仓储层（支持 SQLite / PostgreSQL 无缝切换）"""

    def __init__(self, database_url: str = "sqlite:///:memory:") -> None:
        self.engine = create_engine(database_url, future=True)
        Base.metadata.create_all(self.engine)

    def create_session(self, session_id: str, tenant_id: str, title: str) -> None:
        with Session(self.engine) as db:
            db.add(ChatSession(id=session_id, tenant_id=tenant_id, title=title))
            db.commit()

    def append_message(self, session_id: str, role: str, content: str, tokens: int = 0) -> int:
        with Session(self.engine) as db:
            msg = ChatMessage(session_id=session_id, role=role, content=content, tokens=tokens)
            db.add(msg)
            db.commit()
            db.refresh(msg)
            return msg.id

    def get_history(self, session_id: str, limit: int = 10) -> List[Dict[str, object]]:
        with Session(self.engine) as db:
            stmt = (
                select(ChatMessage)
                .where(ChatMessage.session_id == session_id)
                .order_by(ChatMessage.id.asc())
                .limit(limit)
            )
            rows = db.scalars(stmt).all()
            return [{"role": r.role, "content": r.content, "tokens": r.tokens} for r in rows]

    def write_audit(self, trace_id: str, tenant_id: str, action: str, status: str) -> None:
        with Session(self.engine) as db:
            db.add(AuditLog(trace_id=trace_id, tenant_id=tenant_id, action=action, status=status))
            db.commit()

    def count_audits(self, tenant_id: str) -> int:
        with Session(self.engine) as db:
            rows = db.scalars(select(AuditLog).where(AuditLog.tenant_id == tenant_id)).all()
            return len(rows)


if __name__ == "__main__":
    repo = StarMindRepository()
    repo.create_session("sess-001", "dept_rd", "RAG 检索排查")
    repo.append_message("sess-001", "user", "如何优化检索召回率？", tokens=18)
    repo.append_message("sess-001", "assistant", "建议开启 BM25 + 向量混合检索并叠加 Rerank。", tokens=32)
    repo.write_audit("tr-demo01", "dept_rd", "chat_rag", "ALLOW")
    print("历史消息:", repo.get_history("sess-001"))
    print("审计条数:", repo.count_audits("dept_rd"))


### 🔍 步骤 2 代码深度解析

采用 SQLAlchemy 2.0 强类型 `Mapped` 语法，所有数据库写入均通过 `with Session(self.engine) as db:` 自动提交并释放连接，默认内存 SQLite 让单元测试毫秒级跑完。

**预期运行输出参考：**
```text
历史消息: [{'role': 'user', 'content': '如何优化检索召回率？', 'tokens': 18}, {'role': 'assistant', 'content': '建议开启 BM25 + 向量混合检索并叠加 Rerank。', 'tokens': 32}]
审计条数: 1
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day59/tests/test_day59.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 59 单元测试：配置中心、结构化日志与 SQLAlchemy 仓储底座"""
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import pytest
from pydantic import ValidationError
from src.day59_core_infra import StarMindError, StarMindSettings, format_json_log, new_trace_id
from src.day59_db_repository import StarMindRepository


def test_settings_validation():
    cfg = StarMindSettings(rag_top_k=8, daily_budget_cny=200.0)
    assert cfg.rag_top_k == 8
    with pytest.raises(ValidationError):
        StarMindSettings(daily_budget_cny=-10.0)


def test_json_log_and_error_format():
    tid = new_trace_id()
    raw = format_json_log("warning", "high_latency", tid, {"ms": 1450})
    parsed = json.loads(raw)
    assert parsed["level"] == "WARNING"
    assert parsed["trace_id"] == tid

    err = StarMindError(4001, "参数缺失", 400, tid)
    assert err.to_dict()["code"] == 4001


def test_repository_session_messages_and_audit():
    repo = StarMindRepository("sqlite:///:memory:")
    repo.create_session("s1", "t1", "测试会话")
    m1 = repo.append_message("s1", "user", "你好", 5)
    m2 = repo.append_message("s1", "assistant", "您好，请问有什么可以帮您？", 14)
    assert m2 > m1
    hist = repo.get_history("s1")
    assert len(hist) == 2 and hist[0]["role"] == "user"
    repo.write_audit("tr-1", "t1", "rag_query", "ALLOW")
    assert repo.count_audits("t1") == 1

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：为 Repository 增加按租户统计总 Token 消耗的查询方法

**题目要求：**
编写辅助函数或子类方法，统计某个 `session_id` 下所有消息的累计 `tokens` 总和。

**思路提示：** 调用 `repo.get_history(session_id, limit=1000)` 累加每条记录的 `tokens` 即可。


In [ ]:
from src.day59_db_repository import StarMindRepository


def sum_session_tokens(repo: StarMindRepository, session_id: str) -> int:
    return sum(int(m["tokens"]) for m in repo.get_history(session_id, limit=1000))


r = StarMindRepository()
r.create_session("s1", "t1", "demo")
r.append_message("s1", "user", "hi", 10)
r.append_message("s1", "assistant", "hello", 25)
assert sum_session_tokens(r, "s1") == 35
print("会话总 Token 统计通过: 35")


**答案解析与评分标准：**

会话级 Token 统计是后续实现上下文窗口裁剪和财务对账的基础。


### 作业 2（企业实战）：企业实战：给日志格式化器增加自动密钥字段脱敏（防 api_key 泄露）

**题目要求：**
编写 `safe_json_log`，如果 `extra` 字典中存在键名包含 `key`/`secret`/`token`/`password` 的字段，自动将其值替换为 `***REDACTED***`。

**思路提示：** 遍历 `extra.items()`，对敏感键名做浅拷贝替换后再调用 `format_json_log`。


In [ ]:
from src.day59_core_infra import format_json_log

SENSITIVE_KEYS = ("key", "secret", "token", "password")


def safe_json_log(level: str, event: str, trace_id: str, extra: dict) -> str:
    clean = {
        k: ("***REDACTED***" if any(s in k.lower() for s in SENSITIVE_KEYS) else v)
        for k, v in extra.items()
    }
    return format_json_log(level, event, trace_id, clean)


out = safe_json_log("info", "llm_call", "tr-1", {"model": "qwen", "api_key": "sk-123456"})
assert "sk-123456" not in out and "***REDACTED***" in out
print("安全日志脱敏通过:", out)


**答案解析与评分标准：**

防止调试日志把生产 API Key 打到日志平台是大厂安全红线。


### 作业 3（架构进阶）：架构进阶：实现基于 contextvars 的异步协程级 trace_id 上下文传递

**题目要求：**
使用 Python 标准库 `contextvars.ContextVar` 实现 `trace_id_ctx`，保证高并发 `asyncio.gather` 时不同协程之间的 `trace_id` 互不串台。

**思路提示：** 在每个异步任务入口调用 `trace_id_ctx.set(tid)`，`await asyncio.sleep(...)` 后读取仍保持各自独立值。


In [ ]:
import asyncio
from contextvars import ContextVar

trace_id_ctx: ContextVar[str] = ContextVar("trace_id", default="tr-none")


async def handle_req(req_id: str) -> str:
    trace_id_ctx.set(f"tr-{req_id}")
    await asyncio.sleep(0.01)
    return trace_id_ctx.get()


async def main():
    res = await asyncio.gather(handle_req("A"), handle_req("B"), handle_req("C"))
    assert res == ["tr-A", "tr-B", "tr-C"]
    print("ContextVar 并发 trace_id 隔离验证通过:", res)


asyncio.run(main())


**答案解析与评分标准：**

`ContextVar` 是 FastAPI/Starlette 异步中间件传递请求级链路追踪 ID 的标准底层机制。


---

## 🎯 老陈收尾复盘

地基验收完毕！配置强校验、结构化 JSON 日志带全链路 trace_id、SQLAlchemy 四张核心表 CRUD 全绿通过。今晚提交 `feat/day59-capstone-infra-base`，明天直接上核心混合检索 RAG！
